# 🔍 Transfermarkt Scraper v2
## Búsqueda mejorada por apellido + equipo + edad

**Mejoras en esta versión:**
- Extrae el apellido de "J. Hurtado" → "Hurtado"
- Verifica el equipo para confirmar el jugador correcto
- Usa la edad para validar

---

In [ ]:
# ============================================
# CELDA 1: Instalar dependencias
# ============================================
!pip install requests beautifulsoup4 pandas lxml -q
print("✅ Dependencias instaladas!")

In [ ]:
# ============================================
# CELDA 2: Imports y configuración
# ============================================
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from urllib.parse import quote
import random

# Headers para simular navegador real
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'es-ES,es;q=0.9,en;q=0.8',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Referer': 'https://www.transfermarkt.com/',
}

# Crear sesión para mantener cookies
session = requests.Session()
session.headers.update(HEADERS)

print("✅ Configuración cargada!")

In [ ]:
# ============================================
# CELDA 3: Funciones auxiliares
# ============================================

def extraer_apellido(nombre_completo):
    """
    Extrae el apellido de diferentes formatos:
    - 'J. Hurtado' → 'Hurtado'
    - 'Juan Hurtado' → 'Hurtado'
    - 'J. Hurtado García' → 'Hurtado García'
    - 'Hurtado' → 'Hurtado'
    """
    if pd.isna(nombre_completo):
        return None
    
    nombre = str(nombre_completo).strip()
    
    # Si tiene formato "X. Apellido" o "X. Y. Apellido"
    # Remover iniciales al principio
    patron_inicial = r'^([A-Z]\.\s*)+'
    nombre_sin_inicial = re.sub(patron_inicial, '', nombre).strip()
    
    if nombre_sin_inicial:
        return nombre_sin_inicial
    
    # Si no matcheó el patrón, intentar extraer última(s) palabra(s)
    partes = nombre.split()
    if len(partes) >= 2:
        # Asumir que el primer elemento es nombre, el resto apellido
        return ' '.join(partes[1:])
    
    return nombre


def normalizar_equipo(equipo):
    """
    Normaliza nombre de equipo para comparación
    'Patriotas Boyacá' → ['patriotas', 'boyaca']
    """
    if pd.isna(equipo):
        return []
    
    equipo_lower = str(equipo).lower()
    # Remover acentos básicos
    reemplazos = {'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u', 'ñ': 'n'}
    for k, v in reemplazos.items():
        equipo_lower = equipo_lower.replace(k, v)
    
    # Palabras significativas (más de 3 letras)
    palabras = [p for p in equipo_lower.split() if len(p) > 3]
    return palabras


def equipos_coinciden(equipo_csv, equipo_tm):
    """
    Verifica si dos equipos coinciden (match parcial)
    """
    palabras_csv = normalizar_equipo(equipo_csv)
    equipo_tm_lower = str(equipo_tm).lower()
    
    # Remover acentos del texto TM
    reemplazos = {'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u', 'ñ': 'n'}
    for k, v in reemplazos.items():
        equipo_tm_lower = equipo_tm_lower.replace(k, v)
    
    # Si alguna palabra significativa está en el texto de TM
    for palabra in palabras_csv:
        if palabra in equipo_tm_lower:
            return True
    
    return False


def edad_coincide(edad_csv, edad_tm, tolerancia=1):
    """
    Verifica si las edades coinciden (con tolerancia)
    """
    try:
        edad_csv_int = int(edad_csv)
        edad_tm_int = int(re.search(r'\d+', str(edad_tm)).group())
        return abs(edad_csv_int - edad_tm_int) <= tolerancia
    except:
        return True  # Si no podemos comparar, no descartar


# Test
print("Test extraer_apellido:")
print(f"  'J. Hurtado' → '{extraer_apellido('J. Hurtado')}'")
print(f"  'A. Mejía' → '{extraer_apellido('A. Mejía')}'")
print(f"  'J. C. García' → '{extraer_apellido('J. C. García')}'")
print(f"  'Juan Pérez' → '{extraer_apellido('Juan Pérez')}'")
print("\n✅ Funciones auxiliares listas!")

In [ ]:
# ============================================
# CELDA 4: Funciones de scraping mejoradas
# ============================================

def buscar_jugador_mejorado(nombre_original, equipo, edad=None):
    """
    Busca un jugador en Transfermarkt usando apellido + verificación de equipo/edad
    Retorna: (url, nombre_encontrado) o (None, None)
    """
    try:
        # Extraer apellido para búsqueda
        apellido = extraer_apellido(nombre_original)
        if not apellido:
            return None, None
        
        # Búsqueda en Transfermarkt
        apellido_encoded = quote(apellido)
        search_url = f"https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche?query={apellido_encoded}"
        
        response = session.get(search_url, timeout=15)
        
        if response.status_code != 200:
            return None, None
        
        soup = BeautifulSoup(response.content, 'lxml')
        
        # Buscar tabla de jugadores
        tables = soup.find_all('table', class_='items')
        
        candidatos = []
        
        for table in tables:
            rows = table.find_all('tr', class_=['odd', 'even'])
            
            for row in rows:
                # Obtener link y nombre del jugador
                link_tag = row.find('a', class_='spielprofil_tooltip')
                if not link_tag or not link_tag.get('href'):
                    continue
                
                player_url = "https://www.transfermarkt.com" + link_tag['href']
                player_name = link_tag.get_text(strip=True)
                
                # Obtener equipo de la fila
                row_text = row.get_text()
                
                # Obtener edad si está disponible
                edad_cell = row.find('td', class_='zentriert')
                edad_tm = edad_cell.get_text(strip=True) if edad_cell else None
                
                # Calcular score de coincidencia
                score = 0
                
                # +2 si el equipo coincide
                if equipo and equipos_coinciden(equipo, row_text):
                    score += 2
                
                # +1 si la edad coincide
                if edad and edad_tm and edad_coincide(edad, edad_tm):
                    score += 1
                
                candidatos.append({
                    'url': player_url,
                    'nombre': player_name,
                    'score': score,
                    'row_text': row_text[:100]
                })
        
        if not candidatos:
            return None, None
        
        # Ordenar por score (mayor primero)
        candidatos.sort(key=lambda x: x['score'], reverse=True)
        
        # Retornar el mejor candidato
        mejor = candidatos[0]
        return mejor['url'], mejor['nombre']
    
    except Exception as e:
        print(f"Error en búsqueda: {e}")
        return None, None


def obtener_datos_jugador(url):
    """
    Extrae datos del perfil de un jugador
    """
    datos = {
        'transfermarkt_url': url,
        'nombre_tm': None,
        'valor_mercado': None,
        'agente': None,
        'fin_contrato': None,
        'imagen_url': None
    }
    
    try:
        response = session.get(url, timeout=15)
        
        if response.status_code != 200:
            return datos
        
        soup = BeautifulSoup(response.content, 'lxml')
        
        # Nombre completo del jugador
        header = soup.find('h1', class_='data-header__headline-wrapper')
        if header:
            datos['nombre_tm'] = header.get_text(strip=True)
        
        # 1. Valor de mercado
        market_value = soup.find('a', class_='data-header__market-value-wrapper')
        if market_value:
            datos['valor_mercado'] = market_value.get_text(strip=True)
        
        # 2. Imagen del jugador
        img_tag = soup.find('img', class_='data-header__profile-image')
        if img_tag and img_tag.get('src'):
            datos['imagen_url'] = img_tag['src']
        
        # 3. Buscar información adicional en diferentes secciones
        
        # Método 1: Buscar en info-table
        info_table = soup.find('div', class_='info-table')
        if info_table:
            rows = info_table.find_all(['span', 'a'])
            text_content = info_table.get_text()
            
            # Buscar agente
            agent_link = info_table.find('a', href=re.compile(r'/berater/'))
            if agent_link:
                datos['agente'] = agent_link.get_text(strip=True)
        
        # Método 2: Buscar en data-header
        data_header = soup.find('div', class_='data-header__details')
        if data_header:
            # Buscar contrato
            contract_match = re.search(r'Contract expires[:\s]*(\w+\s+\d+,\s+\d{4}|\d{2}/\d{2}/\d{4})', 
                                       data_header.get_text(), re.I)
            if contract_match:
                datos['fin_contrato'] = contract_match.group(1)
        
        # Método 3: Buscar en toda la página
        if not datos['fin_contrato']:
            page_text = soup.get_text()
            contract_patterns = [
                r'Contract expires[:\s]*(\w+\s+\d+,\s+\d{4})',
                r'Contract until[:\s]*(\w+\s+\d+,\s+\d{4})',
                r'Contrato hasta[:\s]*(\d{2}/\d{2}/\d{4})',
            ]
            for pattern in contract_patterns:
                match = re.search(pattern, page_text, re.I)
                if match:
                    datos['fin_contrato'] = match.group(1)
                    break
        
        # Método 4: Buscar agente si no lo encontramos
        if not datos['agente']:
            agent_section = soup.find('span', string=re.compile(r'Agent', re.I))
            if agent_section:
                parent = agent_section.find_parent(['li', 'div', 'tr'])
                if parent:
                    link = parent.find('a')
                    if link:
                        datos['agente'] = link.get_text(strip=True)
        
        return datos
    
    except Exception as e:
        print(f"Error obteniendo datos: {e}")
        return datos

print("✅ Funciones de scraping mejoradas listas!")

In [ ]:
# ============================================
# CELDA 5: Cargar CSV de jugadores
# ============================================

CSV_URL = "https://docs.google.com/spreadsheets/d/e/2PACX-1vSneBjGlw2I3SyXV-uw1V8Cs_O4lbiQw39melKEZJNhunpshakPrn7AZQBN2L8N9Yw_HA-EeVOt3qvf/pub?gid=2002620668&single=true&output=csv"

print("📥 Cargando CSV de jugadores...")
df_jugadores = pd.read_csv(CSV_URL)

print(f"\n✅ CSV cargado!")
print(f"📊 Total de jugadores: {len(df_jugadores)}")
print(f"\n📋 Columnas disponibles:")
print(list(df_jugadores.columns[:10]))

print(f"\n👀 Primeros 5 jugadores:")
display(df_jugadores[['Jugador', 'Liga', 'Equipo', 'Edad']].head())

In [ ]:
# ============================================
# CELDA 6: ⚙️ CONFIGURACIÓN - MODIFICAR AQUÍ
# ============================================

# 🎯 Rango de jugadores a procesar
DESDE_FILA = 0          # Empezar desde esta fila (0 = primero)
HASTA_FILA = 50         # Hasta esta fila (para TODOS usar: len(df_jugadores))

# ⏱️ Pausas entre requests (evitar bloqueos)
PAUSA_MIN = 3           # Segundos mínimos
PAUSA_MAX = 6           # Segundos máximos

# 📝 Nombres de columnas en tu CSV
COL_JUGADOR = 'Jugador'   # Columna con nombre del jugador
COL_EQUIPO = 'Equipo'     # Columna con equipo
COL_LIGA = 'Liga'         # Columna con liga
COL_EDAD = 'Edad'         # Columna con edad

# ============================================
print("⚙️ CONFIGURACIÓN ACTUAL:")
print(f"   Procesar filas: {DESDE_FILA} a {HASTA_FILA}")
print(f"   Total a procesar: {min(HASTA_FILA, len(df_jugadores)) - DESDE_FILA} jugadores")
print(f"   Pausa entre requests: {PAUSA_MIN}-{PAUSA_MAX} segundos")
tiempo_estimado = ((min(HASTA_FILA, len(df_jugadores)) - DESDE_FILA) * (PAUSA_MIN + PAUSA_MAX) / 2) / 60
print(f"   ⏱️ Tiempo estimado: ~{tiempo_estimado:.0f} minutos")

print(f"\n📋 Ejemplo de conversión de nombres:")
for i in range(min(5, len(df_jugadores))):
    nombre = df_jugadores.iloc[i][COL_JUGADOR]
    apellido = extraer_apellido(nombre)
    print(f"   '{nombre}' → buscar: '{apellido}'")

In [ ]:
# ============================================
# CELDA 7: 🚀 EJECUTAR SCRAPING
# ============================================

resultados = []
encontrados = 0
no_encontrados = 0

total = min(HASTA_FILA, len(df_jugadores)) - DESDE_FILA
print(f"🚀 Iniciando scraping de {total} jugadores...")
print(f"📌 Buscando por APELLIDO + verificando EQUIPO y EDAD")
print("=" * 70)

for idx in range(DESDE_FILA, min(HASTA_FILA, len(df_jugadores))):
    row = df_jugadores.iloc[idx]
    nombre = row[COL_JUGADOR]
    equipo = row.get(COL_EQUIPO, '')
    liga = row.get(COL_LIGA, '')
    edad = row.get(COL_EDAD, None)
    
    apellido = extraer_apellido(nombre)
    progreso = idx - DESDE_FILA + 1
    
    print(f"[{progreso}/{total}] {nombre} → buscar '{apellido}' ({equipo})...", end=" ")
    
    try:
        # Buscar jugador con método mejorado
        url, nombre_tm = buscar_jugador_mejorado(nombre, equipo, edad)
        
        if url:
            encontrados += 1
            print(f"✅ {nombre_tm}")
            
            # Obtener datos adicionales
            datos = obtener_datos_jugador(url)
            datos['jugador_original'] = nombre
            datos['equipo_original'] = equipo
            datos['liga_original'] = liga
            datos['edad_original'] = edad
            resultados.append(datos)
            
            valor = datos['valor_mercado'] or 'Sin valor'
            print(f"      💰 {valor}")
        else:
            no_encontrados += 1
            print("❌ No encontrado")
            resultados.append({
                'jugador_original': nombre,
                'equipo_original': equipo,
                'liga_original': liga,
                'edad_original': edad,
                'nombre_tm': None,
                'transfermarkt_url': None,
                'valor_mercado': None,
                'agente': None,
                'fin_contrato': None,
                'imagen_url': None
            })
    
    except Exception as e:
        print(f"⚠️ Error: {e}")
        no_encontrados += 1
        resultados.append({
            'jugador_original': nombre,
            'equipo_original': equipo,
            'liga_original': liga,
            'edad_original': edad,
            'nombre_tm': None,
            'transfermarkt_url': None,
            'valor_mercado': None,
            'agente': None,
            'fin_contrato': None,
            'imagen_url': None
        })
    
    # Pausa aleatoria para no ser bloqueado
    time.sleep(random.uniform(PAUSA_MIN, PAUSA_MAX))

print("=" * 70)
print(f"\n✅ COMPLETADO!")
print(f"   ✅ Encontrados: {encontrados} ({encontrados/total*100:.1f}%)")
print(f"   ❌ No encontrados: {no_encontrados} ({no_encontrados/total*100:.1f}%)")

In [ ]:
# ============================================
# CELDA 8: 📊 Ver resultados
# ============================================

df_resultados = pd.DataFrame(resultados)

# Reordenar columnas
columnas_orden = ['jugador_original', 'nombre_tm', 'equipo_original', 'liga_original',
                  'edad_original', 'valor_mercado', 'transfermarkt_url', 'agente',
                  'fin_contrato', 'imagen_url']
df_resultados = df_resultados[[c for c in columnas_orden if c in df_resultados.columns]]

print("📋 Muestra de resultados encontrados:")
encontrados_df = df_resultados[df_resultados['transfermarkt_url'].notna()]
display(encontrados_df[['jugador_original', 'nombre_tm', 'equipo_original', 'valor_mercado']].head(15))

In [ ]:
# ============================================
# CELDA 9: 💾 Guardar y descargar CSV
# ============================================

nombre_archivo = f'transfermarkt_jugadores_{DESDE_FILA}_a_{HASTA_FILA}.csv'

# Guardar CSV
df_resultados.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
print(f"💾 Archivo guardado: {nombre_archivo}")

# Descargar
from google.colab import files
files.download(nombre_archivo)
print("\n📥 Descargando archivo...")

In [ ]:
# ============================================
# CELDA 10 (OPCIONAL): Ver jugadores no encontrados
# ============================================

print("❌ Jugadores NO encontrados (para buscar manualmente):")
print("=" * 60)

no_encontrados_df = df_resultados[df_resultados['transfermarkt_url'].isna()]
print(f"Total no encontrados: {len(no_encontrados_df)}\n")

for idx, row in no_encontrados_df.iterrows():
    print(f"• {row['jugador_original']} - {row['equipo_original']} ({row.get('liga_original', 'N/A')})")